In [1]:
# Step 1: Load Dataset
from pyspark.sql import SparkSession
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"

# Start Spark Session
spark = SparkSession.builder.appName("LightcastSalaryPrediction").getOrCreate()

# Load Data
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine", "true").option("escape", "\"").csv("./data/lightcast_job_postings.csv")

# Diagnostics (comment this later before submission)
df.printSchema()
df.show(5)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/14 01:47:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/ubuntu/lab08-VidhiSharma2000/data/lightcast_job_postings.csv.

In [2]:
# Drop rows where Salary is NULL
df = df.filter(df.SALARY.isNotNull())


NameError: name 'df' is not defined

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline


In [3]:
# StringIndexer and OneHotEncoder for the categorical variable
indexer = StringIndexer(inputCol="EMPLOYMENT_TYPE_NAME", outputCol="EMPLOYMENT_TYPE_NAME_Indexed")
encoder = OneHotEncoder(inputCol="EMPLOYMENT_TYPE_NAME_Indexed", outputCol="EMPLOYMENT_TYPE_NAME_Encoded")


NameError: name 'StringIndexer' is not defined

In [ ]:
# Assemble all features
assembler = VectorAssembler(
    inputCols=["MIN_YEARS_EXPERIENCE", "MAX_YEARS_EXPERIENCE", "EMPLOYMENT_TYPE_NAME_Encoded"],
    outputCol="features"
)


In [4]:
# Create a pipeline
pipeline = Pipeline(stages=[indexer, encoder, assembler])

# Fit and transform
data = pipeline.fit(df).transform(df)

# Select final dataset
final_data = data.select("SALARY", "features")


NameError: name 'Pipeline' is not defined

In [5]:
# Feature Engineering

# Drop missing salaries
df = df.filter(df.SALARY.isNotNull())

# Import PySpark ML packages
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Define transformations
indexer = StringIndexer(inputCol="EMPLOYMENT_TYPE_NAME", outputCol="EMPLOYMENT_TYPE_NAME_Indexed")
encoder = OneHotEncoder(inputCol="EMPLOYMENT_TYPE_NAME_Indexed", outputCol="EMPLOYMENT_TYPE_NAME_Encoded")
assembler = VectorAssembler(
    inputCols=["MIN_YEARS_EXPERIENCE", "MAX_YEARS_EXPERIENCE", "EMPLOYMENT_TYPE_NAME_Encoded"],
    outputCol="features"
)

# Create Pipeline
pipeline = Pipeline(stages=[indexer, encoder, assembler])

# Fit and transform
data = pipeline.fit(df).transform(df)

# Select final columns
final_data = data.select("SALARY", "features")

# Show example
final_data.show(5)


NameError: name 'df' is not defined

In [6]:
# Train/Test Split

# Randomly split the dataset
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

# Check how many rows are in train/test
print(f"Training Data Rows: {train_data.count()}")
print(f"Testing Data Rows: {test_data.count()}")


NameError: name 'final_data' is not defined

In [7]:
# Import Linear Regression
from pyspark.ml.regression import LinearRegression

# Initialize the Linear Regression model
lr = LinearRegression(
    labelCol="SALARY", 
    featuresCol="features"
)

# Fit the model to the training data
lr_model = lr.fit(train_data)

# Predict on test data
test_results = lr_model.evaluate(test_data)

# Show Predictions vs Actual
predictions = lr_model.transform(test_data)
predictions.select("SALARY", "prediction").show(5)


NameError: name 'train_data' is not defined

In [8]:
# Print Model Metrics

# R² Score
print(f"R²: {test_results.r2}")

# Root Mean Squared Error (RMSE)
print(f"RMSE: {test_results.rootMeanSquaredError}")

# Mean Absolute Error (MAE)
print(f"MAE: {test_results.meanAbsoluteError}")

# Coefficients and Intercept
print("Coefficients:", lr_model.coefficients)
print("Intercept:", lr_model.intercept)


NameError: name 'test_results' is not defined

In [9]:
# Import necessary libraries
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

# Convert predictions to Pandas for easy plotting
predictions_pd = predictions.select("SALARY", "prediction").toPandas()

# Calculate residuals
predictions_pd["residuals"] = predictions_pd["SALARY"] - predictions_pd["prediction"]

# Create a 2x2 subplot grid
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# 1. Predicted vs Actual
sns.scatterplot(x="SALARY", y="prediction", data=predictions_pd, ax=axs[0, 0])
axs[0, 0].plot([predictions_pd.SALARY.min(), predictions_pd.SALARY.max()],
               [predictions_pd.SALARY.min(), predictions_pd.SALARY.max()],
               color='red', linestyle='--')
axs[0, 0].set_title("Predicted vs Actual Salary")

# 2. Residuals vs Predicted
sns.scatterplot(x="prediction", y="residuals", data=predictions_pd, ax=axs[0, 1])
axs[0, 1].axhline(y=0, color='red', linestyle='--')
axs[0, 1].set_title("Residuals vs Predicted Salary")

# 3. Histogram of Residuals
sns.histplot(predictions_pd["residuals"], kde=True, ax=axs[1, 0])
axs[1, 0].set_title("Histogram of Residuals")

# 4. QQ Plot of Residuals
stats.probplot(predictions_pd["residuals"], dist="norm", plot=axs[1, 1])
axs[1, 1].set_title("QQ Plot of Residuals")

plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'scipy'

In [10]:
# Import
import seaborn as sns
import matplotlib.pyplot as plt

# Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x="SALARY", y="prediction", data=predictions_pd)

# Add a red 45-degree line (perfect prediction)
plt.plot([predictions_pd.SALARY.min(), predictions_pd.SALARY.max()],
         [predictions_pd.SALARY.min(), predictions_pd.SALARY.max()],
         color='red', linestyle='--')

plt.title("Predicted vs Actual Salary")
plt.xlabel("Actual Salary")
plt.ylabel("Predicted Salary")
plt.grid(True)
plt.show()


NameError: name 'predictions_pd' is not defined

<Figure size 800x600 with 0 Axes>